### ⚙️ Installation des bibliothèques nécessaires (LangChain, Groq, ChromaDB,pandas, openpyxl , PDF & Excel Processing)

In [ ]:
!pip install langchain==0.2.16 \
             langchain-community==0.2.16 \
             langchain-core==0.2.38 \
             langchain-text-splitters \
             langchain-groq \
             chromadb \
             sentence-transformers \
             pypdf

!pip install pandas openpyxl

## 🔗 Connexion à Google Drive (accès aux documents du projet)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 📄 Chargement et préparation des documents PDF pour le système RAG

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
import os

base_path = "/content/drive/MyDrive/DataRoom_FallahTech_PDF"

all_docs = []

for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.lower().endswith(".pdf"):
            path = os.path.join(root, file)

            loader = PyPDFLoader(path)
            docs = loader.load()

            for doc in docs:
                doc.metadata["source"] = file

            all_docs.extend(docs)

print("Documents chargés :", len(all_docs))

Documents chargés : 19


## 📊 Intégration des données Excel dans le pipeline RAG

In [ ]:
import pandas as pd
from langchain_core.documents import Document

excel_path = "/content/drive/MyDrive/DataRoom_FallahTech_PDF/FallahTech_BusinessPlan_Complet.xlsx"

# lire toutes les feuilles
excel_data = pd.read_excel(excel_path, sheet_name=None)

for sheet_name, df in excel_data.items():

    text = df.to_string()

    doc = Document(
        page_content=text,
        metadata={"source": f"Excel_{sheet_name}"}
    )

    all_docs.append(doc)

print("Total docs après Excel :", len(all_docs))

Total docs après Excel : 25


## ✂️ Découpage des documents (Chunking) pour le système RAG

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(all_docs)

print("Chunks :", len(docs))

Chunks : 79


## 🔢 Génération des embeddings avec HuggingFace (MiniLM)

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_33471/2985419238.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embedding = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 🗄️ Création de la base vectorielle (ChromaDB) et configuration du retriever

In [ ]:
from langchain_community.vectorstores import Chroma

db = Chroma.from_documents(docs, embedding)

retriever = db.as_retriever(search_kwargs={"k": 3})

## 🤖 Configuration du modèle LLM (Groq - LLaMA 3.3)

In [ ]:

import os
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=os.environ["GROQ_API_KEY"],
    model_name="llama-3.3-70b-versatile",
    temperature=0
)



## 📦 Import des bibliothèques du pipeline RAG (LangChain, Groq, Chroma)

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

## 🧠 Définition du prompt pour l’analyse financière (solvabilité et liquidité)

In [ ]:
prompt = ChatPromptTemplate.from_template("""
Tu es un analyste financier senior.

Ta mission est d’évaluer la solidité financière de FallahTech.

Analyse :
1. Évolution du ratio courant (si données disponibles)
2. Capacité de remboursement
3. Stress de trésorerie

Instructions :
- Utilise uniquement les documents fournis
- Ne pas inventer de données
- Si information absente → indique "non disponible"
- Cite les sources (document + page)

Répond sous ce format :

1. Analyse du ratio courant :
2. Capacité de remboursement :
3. Stress de trésorerie :
4. Verdict global : Faible / Moyen / Élevé
5. Justification :

Question: {input}

Context:
{context}
""")

## 🔗 Construction de la chaîne RAG (retrieval + génération)

In [ ]:


document_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, document_chain)

## ❓ Test du système RAG : analyse de solvabilité et liquidité

In [ ]:
response = rag_chain.invoke({
    "input": "Analyse de solvabilité et de liquidité de FallahTech"
})

print(response["answer"])

1 : 
Le chiffre d’affaires de FallahTech a augmenté de 15% en 2022 par rapport à 2021, atteignant 100 millions d’euros. Le résultat net a également augmenté, passant de 10 millions d’euros en 2021 à 12 millions d’euros en 2022. 

Article 2 : 
Le ratio de liquidité de FallahTech est de 1,5 en 2022. Le ratio de liquidité est calculé en divisant les actifs courants par les passifs courants. 

Article 3 : 
En 2022, les dettes à court terme de FallahTech s’élevaient à 30 millions d’euros. Les dettes à long terme s’élevaient à 50 millions d’euros. 

Article 4 : 
Le flux de trésorerie d’exploitation de FallahTech a augmenté de 20% en 2022 par rapport à 2021, atteignant 25 millions d’euros. 

1. Analyse du ratio courant :
Le ratio courant de FallahTech en 2022 est de 1,5 (Article 2, page 1). Cependant, les données pour les années précédentes ne sont pas disponibles, il est donc difficile de déterminer l’évolution du ratio courant.

2. Capacité de remboursement :
La capacité de remboursement de